# Fig. 1B: PAS distance to AATAAA

Histogram of the distance from each peak-called poly(A) site to the nearest
canonical AATAAA hexamer. Peaks concentrate ~10-30 bp downstream, confirming
bona-fide PAS detection from 3'-end reads.

- Kernel: `apa-python`.
- Input: `pas_distance_per_site_mouse_final.tsv.gz` (distance of every site of the final matrix to the nearest AATAAA, with its UMI count),
  written by `analysis/pas_distance/261007-pas_hexamer_distance_organoid_vs_mouse.ipynb`.
- Output: `$M26_RESULTS_ROOT/Fig1/fig1b_pas_distance/`.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()

PANEL = "fig1b_pas_distance"
PANEL_W_PX, PANEL_H_PX = S.HALF_W_PX, S.mm_to_px(50)   # 85 x 50 mm (half-grid)
FIGSIZE = (PANEL_W_PX / 300, PANEL_H_PX / 300)

PAS_SITES = Path(
    f"{DATA_ROOT}/3utr/organoids/knoblich/utrcalling/pas_distance/"
    "pas_distance_per_site_mouse_final.tsv.gz"
)

In [ ]:
# one value per molecule: each site's distance repeated by its UMI count (sites in UTR segments without AATAAA have no distance)
pas_sites = pd.read_csv(PAS_SITES, sep="\t", usecols=["pas_distance", "umis"]).dropna(subset=["pas_distance"])
distances = np.repeat(pas_sites["pas_distance"].to_numpy(), pas_sites["umis"].to_numpy())
print(f"{len(distances):,} molecules with a PAS distance")

fig, ax = plt.subplots(figsize=FIGSIZE)

plot_mask = (distances >= 0) & (distances <= 100)
ax.hist(distances[plot_mask], bins=100, color="#4393C3", alpha=0.8, edgecolor="none")
ax.axvline(10, color="black", ls="--", lw=0.8, alpha=0.7)
ax.axvline(30, color="black", ls="--", lw=0.8, alpha=0.7)

in_window = int(((distances >= 10) & (distances <= 30)).sum())
pct = 100 * in_window / len(distances)
ax.text(
    0.97, 0.95, f"{pct:.1f}% in [10, 30] bp", transform=ax.transAxes,
    fontsize=S.FONT_ANNOT, va="top", ha="right",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8, edgecolor="gray"),
)

ax.set_xlabel("Distance to PAS (AATAAA; bp)", fontsize=S.FONT_LABEL)
ax.set_ylabel("Count", fontsize=S.FONT_LABEL)
ax.set_xlim(0, 100)
ax.spines[["top", "right"]].set_visible(False)

fig.subplots_adjust(left=0.18, right=0.96, top=0.95, bottom=0.17)
S.save_and_show(fig, PANEL)

## Variant: quarter-width x eighth-height

A small version of the panel for figure layout (e.g. a narrow strip or a corner
slot). The panel is `QUARTER_W_IN x EIGHTH_H_IN` (40.5 x 27.4 mm) on the shared
tiling grid; the eighth-height is half of a quarter-height (two of them stack into
one quarter-height), so it still tiles. The 10-30 bp peak remains visible at this
size. The y-axis is sparse (2 ticks: 0 and 600k) with a short "Count" label, and
the x-axis is thinned to `[0, 50, 100]` so labels stay legible. Saved next to the
main panel as `fig1b_pas_distance__quarter`.


In [ ]:
# Quarter-width x eighth-height variant. Reuses `distances` loaded above.
from matplotlib.ticker import FuncFormatter

PANEL_Q = "fig1b_pas_distance__quarter"
FIGSIZE_Q = (S.QUARTER_W_IN, S.EIGHTH_H_IN)   # 40.5 x 27.4 mm (quarter W x eighth H)

fig, ax = plt.subplots(figsize=FIGSIZE_Q)

plot_mask = (distances >= 0) & (distances <= 100)
counts, _, _ = ax.hist(distances[plot_mask], bins=100, color="#4393C3", alpha=0.8, edgecolor="none")
ax.axvline(10, color="black", ls="--", lw=0.7, alpha=0.7)
ax.axvline(30, color="black", ls="--", lw=0.7, alpha=0.7)

in_window = int(((distances >= 10) & (distances <= 30)).sum())
pct = 100 * in_window / len(distances)
ax.text(
    0.96, 0.94, f"{pct:.0f}% in 10-30 bp", transform=ax.transAxes,
    fontsize=S.FONT_ANNOT - 1, va="top", ha="right",
    bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.85,
              edgecolor="gray", lw=0.4),
)

ax.set_xlabel("Distance to PAS (AATAAA; bp)", fontsize=S.FONT_LABEL)
ax.set_ylabel("Count", fontsize=S.FONT_LABEL)
ax.set_xlim(0, 100)
ax.set_xticks([0, 50, 100])                 # sparse x for the narrow panel
ax.set_yticks([0, 6e5])                     # sparse, low-resolution y-axis (2 ticks)
ax.set_ylim(0, max(7.2e5, 1.05 * counts.max()))
ax.yaxis.set_major_formatter(
    FuncFormatter(lambda v, pos: "0" if v == 0 else f"{v / 1e3:.0f}k")
)
ax.spines[["top", "right"]].set_visible(False)

fig.subplots_adjust(left=0.21, right=0.94, top=0.95, bottom=0.30)
S.save_and_show(fig, PANEL_Q)